In [0]:
!pip install -r ../requirements.txt
%restart_python

In [0]:
import pandas as pd

csv_path = "/Volumes/abs_data/default/raw_volume/Emails.csv"
df = pd.read_csv(csv_path)

f_df = df[df["ExtractedBodyText"].str.len() >= 800]

def get_value(row_index):
    return f_df.iloc[row_index]["ExtractedBodyText"]


In [0]:
from shared.llm_utils import llm_basic, llm_structured, format_system_prompt, commit_tokens
from pydantic import BaseModel

class EmailSummary(BaseModel):
    summary: str
    search_phrases: list[str]
    persons_of_interests: list[str]

temp = """ \
You are provided an email to analyse and your task is to extract the following details:
 - detailed summary of the email
 - list of general search_phrases and additional versions of the search phrases which will be used for categorising the email
 - persons and organisations of interests that were noted in the email
output will just include {summary: str, search_phrases: list[str],  persons_of_interests: list[str]}
    """

# print(format_system_prompt(temp, model="gpt-5-mini"))

In [0]:
ROW_NUMBER = 17

value = get_value(ROW_NUMBER)
print(value)

In [0]:
EMAIL_PROMPT = """ \
You are an Expert Email Analyst. Use a concise, formal, and factual tone. Prioritize accuracy, normalization, and brevity.

<instructions>
You will be given the raw text of a single email in the input variable {{EMAIL_TEXT}}. Your task is to extract three items and produce exactly one valid JSON object and nothing else. Do not output any additional text, explanation, code fences, or diagnostics. Internal reasoning is allowed but must never be output.

Follow these steps:
1. Read the entire raw email in {{EMAIL_TEXT}}.
2. Extract and normalize the required fields (summary, search_phrases, persons_of_interests).
3. Validate that the output is a single syntactically valid JSON object with exactly the keys specified below and proper value types.
4. Output that JSON object as the only content of the response.

<input>
{{EMAIL_TEXT}}

<output_schema>
Return exactly one JSON object with these keys and types:
{
  "summary": string,
  "search_phrases": [string],
  "persons_of_interests": [string]
}
- summary: 2–6 concise sentences describing purpose, key points, requested actions, any deadlines/dates, attachments/references, and tone. Keep factual and brief.
- search_phrases: 3–12 concise, lemmatized, normalized strings optimized for semantic/similarity lookup (see rules).
- persons_of_interests: list of distinct person and organization names mentioned or clearly implied. If none, return [].

<rules>
Output rules (mandatory):
- Output ONLY the single JSON object. No preceding/trailing characters, no code blocks, no commentary.
- Use double quotes for keys and string values. Ensure valid JSON (no trailing commas).
- Preserve the key order exactly as in the schema: summary, search_phrases, persons_of_interests.

Normalization rules for search_phrases:
- Lowercase, remove stopwords, use base forms (lemmas), remove punctuation except meaningful hyphens.
- Provide 3–12 distinct phrases. No near-duplicate variants.
- Normalize dates and amounts into common searchable tokens and include both ISO and natural forms where relevant. Examples: "2024-05-31", "may 31 2024", "usd 5000", "5000 usd".
- Include entity-specific short tokens if useful (e.g., product model, policy name).
- Examples of acceptable phrases: "invoice", "payment overdue", "2024-05-31", "usd 5000", "jane doe", "acme corp".

Normalization rules for persons_of_interests:
- Use normalized proper-case (e.g., "John Smith", "Acme Corp").
- Deduplicate exact duplicates; include organizations and individuals mentioned or implied. Do not invent roles unless explicitly present.

Summary rules:
- 2–6 sentences, factual, cover purpose, key points, requested actions, deadlines/dates, attachments/references, and tone.
- Keep concise and avoid subjective language.

Missing or ambiguous data:
- If a field cannot be determined, return an empty list for lists and a short factual summary mentioning missing info. Still adhere to JSON schema.

Validation:
- Before outputting, ensure JSON parses and adheres to the schema.
- Do not output any parsing or validation messages.

</rules>
    """

summarise = llm_structured(user_prompt=value, system_prompt=EMAIL_PROMPT, text_format=EmailSummary, model="gpt-5")
commit_tokens()
print(summarise.summary)

print(summarise.search_phrases)